<a href="https://colab.research.google.com/github/H-TAMURA-9250/EU_M_Math-Repository/blob/main/Ex8_1_8_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#from pandas.io.formats import style

#data加工・処理・分析ライブラリ
import numpy as np
import numpy.random as random
import scipy as sp
import pandas as pd
from pandas import Series, DataFrame
import csv

#可視化ライブラリ
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

#機械学習ライブラリ
import sklearn

%precision 3

import requests, zipfile
import io

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content

auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

auto.columns = ['symboling','normalized-losses','make','fuel-type','aspiration','num-of-doors', 'body-style','drive-xheels','engine-location','wheel-base','length','width','height' ,'Curb-weight','engine-type','num-of-cylinders','engine-size','fuel-system','bore' ,'stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price']

print('自動車データの形式:{}'.format(auto.shape))

# Select only the columns relevant for the requested task: price, width, engine-size
auto = auto[['price','width','engine-size']]

# Handle missing values represented by '?' and drop rows with NaN
auto = auto.replace('?',np.nan).dropna()
print('自動車データの生成:{}'.format(auto.shape))

print('''データ型の確認（型変換前） {} '''.format(auto.dtypes))

# Convert relevant columns to numeric type
auto = auto.assign(price=pd.to_numeric(auto.price))
auto = auto.assign(width=pd.to_numeric(auto.width))
auto = auto.assign(engine_size=pd.to_numeric(auto['engine-size']))

print('''データ型の確認（型変換後） {}'''.format(auto.dtypes))
auto.corr() # Display correlation matrix for the selected columns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Define features (X) and target (y)
X = auto[['width', 'engine-size']]
y = auto['price']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

# Build and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate the model
print('決定関数(train):{:.3f}'.format(model.score(X_train,y_train)))
print('決定関数(test):{:.3f}'.format(model.score(X_test,y_test)))

# Print regression coefficients and intercept
print('\n回帰係数\n{}'.format(pd.Series(model.coef_, index=X.columns)))
print('切片:{:.3f}'.format(model.intercept_))

auto.head()

# Removed the incomplete Ridge regression part as it was not part of the current request.


自動車データの形式:(205, 26)
自動車データの生成:(201, 3)
データ型の確認（型変換前） price           object
width          float64
engine-size      int64
dtype: object 
データ型の確認（型変換後） price            int64
width          float64
engine-size      int64
engine_size      int64
dtype: object
決定関数(train):0.783
決定関数(test):0.778

回帰係数
width          1261.735518
engine-size     109.526787
dtype: float64
切片:-84060.643


,price,width,engine-size,engine_size
0,13495,64.1,130,130
1,16500,64.1,130,130
2,16500,65.5,152,152
3,13950,66.2,109,109
4,17450,66.4,136,136


### ラッソ回帰の実施

ラッソ回帰（Lasso Regression）は、線形回帰モデルにL1正則化を導入したものです。L1正則化項は、モデルの係数（重み）の絶対値の合計を罰則として加えることで、モデルの複雑さを抑えます。これにより、以下のようなメリットがあります。

*   **特徴量の自動選択（スパース性）**: 影響の少ない特徴量の係数を完全にゼロにする傾向があるため、モデルが自動的に重要な特徴量を選択します。
*   **モデルの解釈性の向上**: ゼロになった係数を持つ特徴量はモデルから実質的に除外されるため、モデルがよりシンプルになり、どの特徴量が予測に寄与しているかが分かりやすくなります。

今回は、`sklearn.linear_model.Lasso`クラスを使用し、`random_state=0`を設定して再現性を確保します。`alpha`パラメータは正則化の強度を調整するもので、ここではデフォルト値を使用しますが、必要に応じて調整することでモデルの性能を最適化できます。

In [2]:
from sklearn.linear_model import Lasso

# Lassoモデルの構築と学習
# alphaは正則化の強度を制御するパラメータです。ここではデフォルト値を使用。
# random_stateは再現性のために設定します。
lasso_model = Lasso(random_state=0)
lasso_model.fit(X_train, y_train)

# モデルの評価
print('ラッソ回帰 (train) 決定係数: {:.3f}'.format(lasso_model.score(X_train, y_train)))
print('ラッソ回帰 (test) 決定係数: {:.3f}'.format(lasso_model.score(X_test, y_test)))

# 回帰係数と切片の表示
print('\nラッソ回帰 回帰係数\n{}'.format(pd.Series(lasso_model.coef_, index=X.columns)))
print('ラッソ回帰 切片: {:.3f}'.format(lasso_model.intercept_))

ラッソ回帰 (train) 決定係数: 0.783
ラッソ回帰 (test) 決定係数: 0.778

ラッソ回帰 回帰係数
width          1261.377493
engine-size     109.541760
dtype: float64
ラッソ回帰 切片: -84038.964


・ラッソ回帰（L1正則化）のメリットと特徴
ラッソ回帰では、正則化項に重みの絶対値の和（L1ノルム）を使用します。

メリット
特徴量の自動選択（スパース性）：
不要、または影響度の低い特徴量の重みを完全にゼロにすることができます。これにより、どの変数が予測に重要なのかがひと目で分かるようになります。

モデルの軽量化・解釈性の向上：
重要な特徴量だけが残るため、データ量が削減され、人間が見ても「何が原因でこの予測になったのか」を説明しやすくなります。

デメリット・注意点
相関関係の強い（マルチコを起こしている）特徴量が複数ある場合、その中からランダムに1つだけを選び、他をゼロにしてしまう傾向があります。

・リッジ回帰（L2正則化）のメリットと特徴
リッジ回帰では、正則化項に重みの二乗和（L2ノルム）を使用します。

メリット
多重共線性（マルチコ）への強さ：
相関関係の強い特徴量が複数存在する場合でも、どれか一つを排除するのではなく、グループ全体の重みを均等に小さく滑らかにする性質があります。

予測精度の安定化：
重みをゼロにはしないため、すべての特徴量の情報を少しずつ残しながら、モデルの極端な動きを滑らかに抑制します。一般的に、予測精度そのものはラッソよりも安定しやすい傾向にあります。

デメリット・注意点
重みは限りなくゼロに近づきますが、完全にゼロにはならないため、特徴量の絞り込み（次元削減）の効果はありません。
